# Count Number of Nodes

In [18]:
import json

with open('mention_networks/network_mega_merge.json', 'r', encoding='utf-8') as f:
    edges = json.load(f)

unique_nodes = set()    # Sets only store unique values
for source, target in edges:
    unique_nodes.add(source)
    unique_nodes.add(target)

print(f"Number of unique nodes: {len(unique_nodes)}")

Number of unique nodes: 10918


# Interactive Graph

In [ ]:
import json
import networkx as nx
import plotly.graph_objects as go

# Load your edges from JSON or define them here
with open("mention_networks/network_mega_merge.json", "r", encoding="utf-8") as f:
    edges = json.load(f)

# Build directed graph
G = nx.DiGraph()
G.add_edges_from(edges)

start_user = "eldonutjaja"

# Compute shortest path length from start_user
depths = dict(nx.single_source_shortest_path_length(G, start_user))

# Compute positions
pos = nx.spring_layout(G, k=0.5, seed=42)

# Extract node positions
x_nodes = [pos[node][0] for node in G.nodes()]
y_nodes = [pos[node][1] for node in G.nodes()]

# Edge traces
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

# Node sizes based on in-degree (number of incoming edges)
node_sizes = []
for node in G.nodes():
    indegree = G.in_degree(node)
    size = 10 + 5 * indegree  # Adjust base size and scaling factor as needed
    node_sizes.append(size)


# Node trace
node_trace = go.Scatter(
    x=x_nodes, y=y_nodes,
    mode='markers+text',
    text=[str(node) for node in G.nodes()],
    textposition="top center",
    hoverinfo='text',
    marker=dict(
        showscale=False,
        color=["lightcoral" if node == start_user else "skyblue" for node in G.nodes()],
        size=node_sizes,
        line_width=2))

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title="Interactive Twitter Mention Network",
                    title_font_size=16,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20, l=5, r=5, t=40),
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.show()


# Pahtfinder between two accounts

In [ ]:
def find_and_visualize_path(G, source_user, target_user):
    try:
        # Find the shortest path
        path = nx.shortest_path(G, source=source_user, target=target_user)
        print(f"✅ Path found from @{source_user} to @{target_user}:")
        print(" → ".join(path))
    except nx.NetworkXNoPath:
        print(f"❌ No path found from @{source_user} to @{target_user}.")
        return
    except nx.NodeNotFound as e:
        print(f"❌ Node not found: {e}")
        return

    # Create subgraph from path
    path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    path_graph = nx.DiGraph()
    path_graph.add_edges_from(path_edges)

    # Layout for consistent node positions
    pos = nx.spring_layout(path_graph, k=0.5, seed=42)

    x_nodes = [pos[node][0] for node in path_graph.nodes()]
    y_nodes = [pos[node][1] for node in path_graph.nodes()]

    edge_x = []
    edge_y = []
    for edge in path_graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=2, color='black'),
        hoverinfo='none',
        mode='lines')

    # Color and size nodes
    node_trace = go.Scatter(
        x=x_nodes, y=y_nodes,
        mode='markers+text',
        text=[str(node) for node in path_graph.nodes()],
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=["lightcoral" if node == source_user else "orange" if node == target_user else "lightgreen" for node in path_graph.nodes()],
            size=[25 for _ in path_graph.nodes()],
            line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=f"Path from @{source_user} to @{target_user}",
                        title_font_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()


In [19]:
find_and_visualize_path(G, "rockstargames", "korn")

✅ Path found from @rockstargames to @korn:
rockstargames → xboxgamepass → expedition33 → xcuculus → linkinpark → rockvillefest → korn


# Visualize User Network

In [ ]:
def visualize_user_network(G, user, max_depth=3):
    if user not in G:
        print(f"❌ User @{user} not found in the network.")
        return

    # Use BFS to find all reachable nodes up to max_depth
    bfs_edges = list(nx.bfs_edges(G, user, depth_limit=max_depth))
    sub_nodes = {user}
    for u, v in bfs_edges:
        sub_nodes.add(u)
        sub_nodes.add(v)

    # Create the subgraph
    subgraph = G.subgraph(sub_nodes).copy()

    print(f"📌 Showing local mention network for @{user} (depth: {max_depth}) with {len(subgraph.nodes)} nodes.")

    # Layout positions
    pos = nx.spring_layout(subgraph, k=0.5, seed=42)
    x_nodes = [pos[node][0] for node in subgraph.nodes()]
    y_nodes = [pos[node][1] for node in subgraph.nodes()]

    # Edge traces
    edge_x = []
    edge_y = []
    for edge in subgraph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1.5, color='gray'),
        hoverinfo='none',
        mode='lines'
    )

    # Node trace
    node_trace = go.Scatter(
        x=x_nodes, y=y_nodes,
        mode='markers+text',
        text=[str(node) for node in subgraph.nodes()],
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=["lightcoral" if node == user else "skyblue" for node in subgraph.nodes()],
            size=[25 if node == user else 15 for node in subgraph.nodes()],
            line_width=2
        )
    )

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=f"Local Network of @{user} (up to depth {max_depth})",
                        title_font_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()


In [ ]:
visualize_user_network(G, "elonmusk", max_depth=4)

# Visualize Node Parents (Incoming Network)

In [ ]:
import json
import networkx as nx
import plotly.graph_objects as go
from collections import deque

def visualize_incoming_graph(json_path, target_user, max_depth=1):
    # Load edges from file
    with open(json_path, "r", encoding="utf-8") as f:
        edges = json.load(f)

    # Build directed graph
    G = nx.DiGraph()
    G.add_edges_from(edges)

    if target_user not in G.nodes:
        print(f"User '{target_user}' not found in the graph.")
        return

    # BFS in reverse direction to collect nodes within `max_depth` levels pointing to target_user
    visited = set()
    sub_edges = []
    queue = deque([(target_user, 0)])

    while queue:
        current_node, depth = queue.popleft()
        if depth > max_depth:
            continue
        for predecessor in G.predecessors(current_node):
            edge = (predecessor, current_node)
            if edge not in sub_edges:
                sub_edges.append(edge)
            if predecessor not in visited:
                visited.add(predecessor)
                queue.append((predecessor, depth + 1))

    # Create subgraph
    sub_nodes = set([u for u, v in sub_edges] + [v for u, v in sub_edges])
    H = G.subgraph(sub_nodes).copy()

    # Compute layout
    pos = nx.spring_layout(H, k=0.5, seed=42)

    # Edge trace
    edge_x = []
    edge_y = []
    for u, v in H.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1, color='#888'),
        hoverinfo='none',
        mode='lines')

    # Node sizes and colors
    node_sizes = [15 if node == target_user else 10 for node in H.nodes()]
    node_colors = ["lightcoral" if node == target_user else "skyblue" for node in H.nodes()]

    node_trace = go.Scatter(
        x=[pos[n][0] for n in H.nodes()],
        y=[pos[n][1] for n in H.nodes()],
        mode='markers+text',
        text=[str(n) for n in H.nodes()],
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=node_colors,
            size=node_sizes,
            line_width=2)
    )

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=f"Incoming Mentions to @{target_user} (depth={max_depth})",
                        title_font_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()

In [ ]:
if __name__ == "__main__":
    json_file = "mention_networks/network_mega_merge.json"
    target = "korn"
    visualize_incoming_graph(json_file, target_user=target, max_depth=5)  # Change depth here